[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.7 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )


    def causal_self_attention(self, x):   # multi-head self-attention
        # As of now shapes must be (B, S, D)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        B, S, D = Q.shape

        # (B, n_h, S, d_k)
        q = Q.view(B, S, self.num_heads, self.head_dim).transpose(1,2)
        k = K.view(B, S, self.num_heads, self.head_dim).transpose(1,2)
        v = V.view(B, S, self.num_heads, self.head_dim).transpose(1,2)

        # upon k.transpose(-1, -2) -> (B, n_h, d_k, S)
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)

        mask = torch.triu(torch.ones(B, self.num_heads, S, S), diagonal=1).bool()
        masked_scores = scores.masked_fill(mask, float('-inf'))
        # (B, n_h, S, S)
        weights = torch.softmax(masked_scores, dim=-1)

        # (B, n_h, S, S) @ (B, n_h, S, d_k) -> (B, n_h, S, d_k)
        attn_output = torch.matmul(weights, v)

        # (B, S, n_h, d_k)
        attn_output = attn_output.transpose(1,2).contiguous()
        # (B, S, D)
        x = attn_output.view(B, S, D)

        return self.W_o(x)

    def forward(self, x):

        x = x + self.causal_self_attention(self.ln1(x))
        x = x + self.mlp(self.ln2(x))

        return x

In [7]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

Output shape: torch.Size([2, 8, 64])
Is nn.Module? True
Params: 49984


In [8]:
from torch_judge import check
check('gpt2_block')


🧪 Testing: GPT-2 Transformer Block (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (4.5ms)
  ✅ [2/5] Has LayerNorm (pre-norm architecture) (2.0ms)
  ✅ [3/5] MLP has 4x expansion with GELU (1.7ms)
  ✅ [4/5] Causal masking — future doesn't affect past (31.8ms)
  ✅ [5/5] Gradient flow to all parameters (34.4ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (74.4ms total)
  Progress saved. Run status() to see your dashboard.

